# CascadeNet Model Architecture

A 1D Convolutional Neural Network (CNN) for spike rate inference from calcium imaging dF/F traces. This notebook documents the architecture used by the CascadeTorch project, walking through each layer, tensor shapes, and configuration options.

In [ ]:
import sys, os
# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
from src.models.components.cascade_net import CascadeNet

# Optional visualization
try:
    from torchinfo import summary
    HAS_TORCHINFO = True
except ImportError:
    HAS_TORCHINFO = False
    print("Install torchinfo for detailed model summary: pip install torchinfo")

## Model Overview

- CASCADE uses a 1D CNN that takes a windowed dF/F trace and predicts the spike rate at the center timepoint
- **Architecture:** 3 Conv1d → 2 MaxPool1d → per-timestep Dense → Flatten → Output
- **Default configuration:** ~34.4K parameters, `windowsize=64`

In [ ]:
net = CascadeNet(windowsize=64)
print(net)
print(f"\nTotal parameters: {sum(p.numel() for p in net.parameters()):,}")

## Detailed Architecture Summary

In [ ]:
if HAS_TORCHINFO:
    summary(net, input_size=(1, 64, 1), col_names=["input_size", "output_size", "num_params", "kernel_size"], verbose=2)
else:
    # Manual parameter count per layer
    for name, param in net.named_parameters():
        print(f"{name:30s} {str(list(param.shape)):20s} {param.numel():>8,}")

## Forward Pass Walkthrough

Trace tensor shapes through each layer of the network to understand how the input is transformed into a spike rate prediction.

In [ ]:
# Trace tensor shapes through the network
x = torch.randn(1, 64, 1)
print(f"Input:           {x.shape}  (batch, windowsize, channels)")

# Permute for Conv1d
x_p = x.permute(0, 2, 1)
print(f"After permute:   {x_p.shape}  (batch, channels, windowsize)")

# Conv layers
x1 = net.relu1(net.conv1(x_p))
print(f"After conv1+relu:{x1.shape}  (batch, {net.filter_numbers[0]}, {x1.shape[2]})")

x2 = net.relu2(net.conv2(x1))
print(f"After conv2+relu:{x2.shape}  (batch, {net.filter_numbers[1]}, {x2.shape[2]})")

x3 = net.pool1(x2)
print(f"After pool1:     {x3.shape}  (batch, {net.filter_numbers[1]}, {x3.shape[2]})")

x4 = net.relu3(net.conv3(x3))
print(f"After conv3+relu:{x4.shape}  (batch, {net.filter_numbers[2]}, {x4.shape[2]})")

x5 = net.pool2(x4)
print(f"After pool2:     {x5.shape}  (batch, {net.filter_numbers[2]}, {x5.shape[2]})")

# Per-timestep dense
x6 = x5.permute(0, 2, 1)
print(f"After permute:   {x6.shape}  (batch, time, channels)")

x7 = net.relu4(net.dense1(x6))
print(f"After dense1:    {x7.shape}  (batch, time, {net.dense_expansion})")

x8 = x7.reshape(x7.size(0), -1)
print(f"After flatten:   {x8.shape}  (batch, {x8.shape[1]})")

out = net.dense2(x8)
print(f"Output:          {out.shape}  (batch, 1) = spike rate prediction")

## Architecture Variants

The window size determines the temporal context used for prediction. Different window sizes are appropriate for different imaging frame rates.

| Window Size | Typical Use Case | Approximate Parameters |
|-------------|-----------------|----------------------|
| 32 | Low frame rate | ~18K |
| 64 | Default (30 Hz) | ~34K |
| 128 | High frame rate (>30 Hz) | ~68K |

In [ ]:
variants = [
    {"windowsize": 64, "desc": "Default (30 Hz)"},
    {"windowsize": 128, "desc": "High frame rate (>30 Hz)"},
    {"windowsize": 32, "desc": "Low frame rate"},
]

print(f"{'Variant':<25s} {'Window':>8s} {'Params':>10s} {'Output dim':>12s}")
print("-" * 60)
for v in variants:
    net_v = CascadeNet(windowsize=v["windowsize"])
    n_params = sum(p.numel() for p in net_v.parameters())
    dummy = torch.randn(1, v["windowsize"], 1)
    out = net_v(dummy)
    print(f"{v['desc']:<25s} {v['windowsize']:>8d} {n_params:>10,} {str(list(out.shape)):>12s}")

## Training Configuration

- **Optimizer:** Adagrad (lr=0.05)
- **Loss:** MSE (Mean Squared Error)
- **Ensemble:** 5 models per noise level
- **Noise levels:** 1-9

In [ ]:
from omegaconf import OmegaConf

# Load the default model config
model_cfg = OmegaConf.load(os.path.join(project_root, 'configs/model/cascade.yaml'))
print(OmegaConf.to_yaml(model_cfg))